# Diffusion, for Images and for Text: Learn to Undo a Little Noise

The runnable code of [Diffusion, for Images and for Text: Learn to Undo a Little Noise](https://residual-stream.lvtan71.workers.dev/posts/diffusion-for-images-and-text/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/diffusion-for-images-and-text.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 5 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "diffusion-for-images-and-text"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/diffusion-for-images-and-text/diff2d.py']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## One model, many step counts

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/diffusion-for-images-and-text/#one-model-many-step-counts)

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn

# A diffusion model on "VAEs and GANs"'s test: 8 Gaussian modes on a ring of radius
# 2, std 0.05, scored the same way (modes covered, share of samples within
# 3 std of a mode). One network predicts the noise; it is trained with the
# mode as a label that is dropped 10% of the time, so the same network
# samples unconditionally (label dropped) and with guidance.
centers = torch.tensor([[2 * math.cos(a), 2 * math.sin(a)] for a in np.arange(8) * math.pi / 4], dtype=torch.float32)
def data(n, g):
    c = torch.randint(0, 8, (n,), generator=g)
    return centers[c] + 0.05 * torch.randn(n, 2, generator=g), c
def score(x):
    d = torch.cdist(x, centers)
    near = d.min(1).values < 0.15
    hits = torch.bincount(d.argmin(1)[near], minlength=8)
    return int((hits > 0.01 * len(x)).sum()), near.float().mean().item()

# The forward process: q(x_t | x_0) = N(sqrt(abar_t) x_0, (1 - abar_t) I),
# with the linear schedule of Ho et al. (2020).
T = 1000
beta = torch.linspace(1e-4, 0.02, T, dtype=torch.float64)
abar = torch.cumprod(1 - beta, 0)
print("linear schedule, beta from 1e-4 to 0.02 over T = 1000 steps")
print(f"{'t':>5s} {'abar_t':>9s} {'signal sqrt(abar)':>18s} {'noise sqrt(1-abar)':>19s} {'SNR':>9s}")
for t in (1, 10, 100, 250, 500, 750, 1000):
    a = abar[t - 1].item()
    print(f"{t:5d} {a:9.5f} {math.sqrt(a):18.4f} {math.sqrt(1 - a):19.4f} {a / (1 - a):9.4g}")
abar = abar.float()

class Eps(nn.Module):
    """eps_theta(x_t, t, label): an MLP on the point, a sinusoidal embedding
    of t, and a learned embedding of the label (index 8 = no label)."""
    def __init__(self, h=256):
        super().__init__()
        self.label = nn.Embedding(9, 64)
        self.net = nn.Sequential(nn.Linear(2 + 64 + 64, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t, c):
        f = torch.exp(-math.log(1000) * torch.arange(32) / 32)
        a = t[:, None].float() * f
        return self.net(torch.cat([x, a.sin(), a.cos(), self.label(c)], -1))

def train(seed, steps=20000):
    g = torch.Generator().manual_seed(seed); torch.manual_seed(seed)
    net = Eps(); opt = torch.optim.Adam(net.parameters(), 1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    for _ in range(steps):
        x0, c = data(512, g)
        c = torch.where(torch.rand(512, generator=g) < 0.1, 8, c)             # drop the label 10% of the time
        t = torch.randint(0, T, (512,), generator=g)
        eps = torch.randn(512, 2, generator=g)
        a = abar[t][:, None]
        loss = ((net(a.sqrt() * x0 + (1 - a).sqrt() * eps, t, c) - eps) ** 2).mean()   # the DDPM loss
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return net

@torch.no_grad()
def guided(net, x, t, c, w):
    """Classifier-free guidance: eps_uncond + w (eps_cond - eps_uncond).
    w = 1 is the plain conditional model; c = 8 means unconditional."""
    tt = torch.full((len(x),), t)
    if w == 1 or (c == 8).all():
        return net(x, tt, c)
    e_u = net(x, tt, torch.full_like(c, 8))
    return e_u + w * (net(x, tt, c) - e_u)

@torch.no_grad()
def ddpm(net, n, c=8, w=1.0):
    """Ancestral sampling, all T steps: x_{t-1} = mean + sqrt(beta_t) z."""
    x, c = torch.randn(n, 2), torch.full((n,), c) if isinstance(c, int) else c
    for t in reversed(range(T)):
        e = guided(net, x, t, c, w)
        x = (x - beta[t].float() / (1 - abar[t]).sqrt() * e) / (1 - beta[t].float()).sqrt()
        if t > 0:
            x = x + beta[t].float().sqrt() * torch.randn_like(x)
    return x

@torch.no_grad()
def ddim(net, n, steps, c=8, w=1.0):
    """DDIM with eta = 0 (Song et al., 2021): deterministic, and free to skip
    steps. Predict x_0 from the noise estimate, then jump to the next t."""
    x, c = torch.randn(n, 2), torch.full((n,), c) if isinstance(c, int) else c
    ts = torch.linspace(T - 1, 0, steps).round().long().tolist()
    for i, t in enumerate(ts):
        e = guided(net, x, t, c, w)
        x0 = (x - (1 - abar[t]).sqrt() * e) / abar[t].sqrt()
        a_next = abar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0)
        x = a_next.sqrt() * x0 + (1 - a_next).sqrt() * e
    return x

nets = [train(s) for s in range(3)]
print(f"\nunconditional samples (2,000 per run); modes covered, then share near a mode, per seed")
for name, fn in [("DDPM, 1000 steps", lambda m: ddpm(m, 2000))] + \
                [(f"DDIM, {k} steps", lambda m, k=k: ddim(m, 2000, k)) for k in (100, 50, 20, 10, 5, 2)]:
    torch.manual_seed(100)
    res = [score(fn(m)) for m in nets]
    print(f"{name:18s} {'  '.join(str(r[0]) for r in res):>9s}    {'  '.join(f'{r[1]:4.0%}' for r in res)}", flush=True)

print(f"\nclassifier-free guidance, DDIM 50 steps, 250 samples asked for each mode (2,000 per run)")
print(f"{'w':>4s} {'landed on the mode asked for':>30s} {'share near any mode':>21s}")
asked = torch.arange(8).repeat_interleave(250)
for w in (0.0, 1.0, 3.0, 7.0):
    torch.manual_seed(200)
    on, near = [], []
    for m in nets:
        x = ddim(m, 2000, 50, c=asked, w=w)
        d = torch.cdist(x, centers)
        on.append(((d.argmin(1) == asked) & (d.min(1).values < 0.15)).float().mean().item())
        near.append(score(x)[1])
    print(f"{w:4.0f} {'  '.join(f'{v:4.0%}' for v in on):>30s} {'  '.join(f'{v:4.0%}' for v in near):>21s}", flush=True)
torch.save([m.state_dict() for m in nets], "nets.pt")

The post printed:

```text
unconditional samples (2,000 per run); modes covered, then share near a mode, per seed
DDPM, 1000 steps     8  8  8     84%   85%   85%
DDIM, 100 steps      8  8  8     83%   82%   81%
DDIM, 50 steps       8  8  8     81%   80%   80%
DDIM, 20 steps       8  8  8     76%   73%   73%
DDIM, 10 steps       8  8  8     58%   55%   51%
DDIM, 5 steps        0  0  1      3%    3%    3%
DDIM, 2 steps        0  1  0      2%    2%    2%
```

In [ ]:
import numpy as np
import torch
# Is DDIM's collapse at 5 steps about the number of steps, or where they
# fall? diff2d.py's trained models, sampled with three choices of the
# timesteps to visit. diff2d.py's own choice, evenly spaced from 999 down
# to 0, wastes its last step at t = 0 (almost no noise left) and leaves
# one jump from t = 250 to clean data.
src = open("diff2d.py").read().split("nets = [train(s)")[0]
ns = {}; exec(src.replace("print(", "(lambda *a, **k: None)("), ns)
nets = []
for sd in torch.load("nets.pt"):
    m = ns["Eps"](); m.load_state_dict(sd); nets.append(m)
abar, score = ns["abar"], ns["score"]

@torch.no_grad()
def ddim_at(net, n, ts):
    """diff2d.py's ddim(), visiting the given timesteps (descending)."""
    x, c = torch.randn(n, 2), torch.full((n,), 8)
    for i, t in enumerate(ts):
        e = ns["guided"](net, x, t, c, 1.0)
        x0 = (x - (1 - abar[t]).sqrt() * e) / abar[t].sqrt()
        a_next = abar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0)
        x = a_next.sqrt() * x0 + (1 - a_next).sqrt() * e
    return x

spacings = {
    "even, ending at t = 0 (diff2d.py)": lambda k: torch.linspace(999, 0, k).round().long().tolist(),
    "even, ending at t = 999/k": lambda k: [round(999 - i * 1000 / k) for i in range(k)],
    "quadratic (DDIM paper)": lambda k: sorted({round(999 * (1 - i / k) ** 2) for i in range(k)}, reverse=True),
}
print("unconditional, 2,000 samples per seed: share near a mode (modes covered)")
for k in (5, 10, 20):
    for name, f in spacings.items():
        ts = f(k)
        torch.manual_seed(100)
        res = [score(ddim_at(m, 2000, ts)) for m in nets]
        print(f"{k:3d} steps, {name:34s} {'  '.join(f'{r[1]:4.0%} ({r[0]})' for r in res)}   t = {ts}"
              if k == 5 else
              f"{k:3d} steps, {name:34s} {'  '.join(f'{r[1]:4.0%} ({r[0]})' for r in res)}", flush=True)

The post printed:

```text
unconditional, 2,000 samples per seed: share near a mode (modes covered)
  5 steps, even, ending at t = 0 (diff2d.py)    3% (0)    3% (0)    3% (1)   t = [999, 749, 500, 250, 0]
  5 steps, even, ending at t = 999/k           17% (8)   13% (7)   12% (8)   t = [999, 799, 599, 399, 199]
  5 steps, quadratic (DDIM paper)              50% (8)   42% (8)   39% (8)   t = [999, 639, 360, 160, 40]
 10 steps, even, ending at t = 0 (diff2d.py)   58% (8)   55% (8)   51% (8)
 10 steps, even, ending at t = 999/k           61% (8)   58% (8)   55% (8)
 10 steps, quadratic (DDIM paper)              67% (8)   65% (8)   63% (8)
 20 steps, even, ending at t = 0 (diff2d.py)   76% (8)   73% (8)   73% (8)
 20 steps, even, ending at t = 999/k           76% (8)   74% (8)   75% (8)
 20 steps, quadratic (DDIM paper)              77% (8)   74% (8)   74% (8)
```

## Classifier-free guidance

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/diffusion-for-images-and-text/#classifier-free-guidance)

In [ ]:
import math
import torch
# Where guided samples go: for each sample, its distance from the center
# of the ring (the true modes sit at radius 2) and its spread around the
# mode it was asked for (the data's own spread is 0.05 per coordinate).
src = open("diff2d.py").read().split("nets = [train(s)")[0]
ns = {}; exec(src.replace("print(", "(lambda *a, **k: None)("), ns)
nets = []
for sd in torch.load("nets.pt"):
    m = ns["Eps"](); m.load_state_dict(sd); nets.append(m)
asked = torch.arange(8).repeat_interleave(250)
print("DDIM 50 steps, 250 samples per mode, all three seeds pooled")
print(f"{'w':>3s} {'median radius':>14s} {'spread around the mode asked for':>33s}")
for w in (1.0, 3.0, 7.0):
    torch.manual_seed(200)
    xs = torch.cat([ns["ddim"](m, 2000, 50, c=asked, w=w) for m in nets])
    target = ns["centers"][asked.repeat(3)]
    print(f"{w:3.0f} {xs.norm(dim=1).median():14.2f} {(xs - target).std(0).mean():33.3f}")

The post printed:

```text
DDIM 50 steps, 250 samples per mode, all three seeds pooled
  w  median radius  spread around the mode asked for
  1           2.00                             0.032
  3           2.06                             0.081
  7           3.98                             4.737
```

## What parallel decoding gets wrong

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/diffusion-for-images-and-text/#what-parallel-decoding-gets-wrong)

In [ ]:
import torch
import torch.nn as nn

# Masked diffusion for text, on a toy language where every token depends
# on the others: the 90 sequences x_i = (a + i d) mod 10, i = 0..7, for a
# start a in 0..9 and a step d in 1..9 ("3 6 9 2 5 8 1 4"). Two adjacent
# digits pin down the rest. Token 10 is [MASK].
L, MASK = 8, 10
lang = torch.tensor([[(a + i * d) % 10 for i in range(L)] for a in range(10) for d in range(1, 10)])
valid = {tuple(r) for r in lang.tolist()}

class Denoiser(nn.Module):
    """A small bidirectional Transformer: every position sees every other,
    masked or not, and predicts a digit for each."""
    def __init__(self, d=64):
        super().__init__()
        self.tok, self.pos = nn.Embedding(11, d), nn.Parameter(0.02 * torch.randn(L, d))
        layer = nn.TransformerEncoderLayer(d, 4, 4 * d, dropout=0.0, batch_first=True)
        self.body, self.out = nn.TransformerEncoder(layer, 3), nn.Linear(d, 10)
    def forward(self, x):
        return self.out(self.body(self.tok(x) + self.pos))

def train(seed, steps=10000):
    """The masked-diffusion loss (MDLM, Sahoo et al. 2024): draw a masking
    rate t, mask each token with probability t, and score the model's
    cross-entropy on the masked tokens, weighted by 1/t."""
    torch.manual_seed(seed)
    net = Denoiser(); opt = torch.optim.Adam(net.parameters(), 1e-3)
    for _ in range(steps):
        x0 = lang[torch.randint(0, len(lang), (256,))]
        t = torch.rand(256, 1).clamp(min=1 / L)
        m = torch.rand(256, L) < t
        logits = net(torch.where(m, MASK, x0))
        ce = nn.functional.cross_entropy(logits.transpose(1, 2), x0, reduction="none")
        loss = ((ce * m) / t).sum(1).mean() / L
        opt.zero_grad(); loss.backward(); opt.step()
    return net

@torch.no_grad()
def sample(net, n, k, order):
    """Start fully masked; each step unmasks k positions (k can be a list,
    one entry per step), sampling each from the model's distribution for
    that position, independently of the others unmasked in the same step."""
    x = torch.full((n, L), MASK)
    for k in (k if isinstance(k, list) else [k] * (L // k)):
        probs = net(x).softmax(-1)
        draw = torch.distributions.Categorical(probs).sample()
        free = x == MASK
        if order == "left to right":
            key = -torch.arange(L).float().expand(n, L)
        elif order == "random":
            key = torch.rand(n, L)
        else:                                               # "most confident": highest top probability first
            key = probs.max(-1).values
        pick = torch.where(free, key, -float("inf")).topk(k, -1).indices
        x.scatter_(1, pick, draw.gather(1, pick))
    return x

import itertools
def first_step_exact(k, order):
    """With everything masked, a perfect model's distribution at each
    position is uniform over the 10 digits, and one step samples the k
    chosen positions independently. The chance those k digits fit some
    valid sequence: valid patterns on those positions / 10^k, averaged
    over which positions the order picks."""
    sets = [tuple(range(k))] if order == "left to right" else list(itertools.combinations(range(L), k))
    return sum(len({tuple(r[i] for i in p) for r in lang.tolist()}) / 10 ** k for p in sets) / len(sets)

nets = [train(s) for s in range(3)]
print(f"{len(lang)} valid sequences of {L} digits; 3 models (seeds), 2,000 samples each\n")
print(f"{'tokens per step':>15s} {'steps':>6s} {'order':>15s}   valid samples, per seed     distinct valid (of 90)   first step, exact")
for k in (1, 2, 4, 8):
    for order in (("left to right", "random", "most confident") if k < L else ("all at once",)):
        torch.manual_seed(1)
        res = []
        for net in nets:
            s = [tuple(r) for r in sample(net, 2000, k, order).tolist()]
            ok = [r for r in s if r in valid]
            res.append((len(ok) / len(s), len(set(ok))))
        exact = f"{first_step_exact(k, 'random' if order == 'all at once' else order):8.2%}" if order != "most confident" else ""
        print(f"{k:15d} {L // k:6d} {order:>15s}   {'  '.join(f'{v:6.1%}' for v, _ in res)}      "
              f"{'  '.join(f'{d:3d}' for _, d in res)}   {exact:>16s}", flush=True)

# Parallel is safe once the context determines the rest: two digits one at
# a time, then the other six in a single step.
for order in ("left to right", "random"):
    torch.manual_seed(1)
    res = []
    for net in nets:
        s = [tuple(r) for r in sample(net, 2000, [1, 1, 6], order).tolist()]
        ok = [r for r in s if r in valid]
        res.append((len(ok) / len(s), len(set(ok))))
    print(f"{'1, 1, then 6':>15s} {3:6d} {order:>15s}   {'  '.join(f'{v:6.1%}' for v, _ in res)}      "
          f"{'  '.join(f'{d:3d}' for _, d in res)}", flush=True)

# Why random order fails here: two digits at distance g fix the step d only
# when g d = (difference) mod 10 has one solution, i.e. g is coprime to 10.
for g in range(1, L):
    n = sum(len({d for d in range(1, 10) if (g * d) % 10 == e}) == 1 for e in range(10)
            if any((g * d) % 10 == e for d in range(1, 10)))
    m = sum(any((g * d) % 10 == e for d in range(1, 10)) for e in range(10))
    print(f"digits {g} apart: {n} of the {m} possible differences fix the step", flush=True)

The post printed:

```text
90 valid sequences of 8 digits; 3 models (seeds), 2,000 samples each

tokens per step  steps           order   valid samples, per seed     distinct valid (of 90)   first step, exact
              1      8   left to right    99.1%   98.5%   98.7%       90   90   90            100.00%
              1      8          random    99.1%   98.9%   98.3%       90   90   90            100.00%
              1      8  most confident    99.9%   99.8%   99.9%       90   90   90                   
              2      4   left to right    90.1%   89.5%   89.2%       90   90   90             90.00%
              2      4          random    58.1%   58.8%   58.2%       90   90   90             65.36%
              2      4  most confident    30.4%   88.4%   89.2%       88   90   90                   
              4      2   left to right     1.1%    0.9%    1.0%       22   19   18              0.90%
              4      2          random     1.1%    0.9%    0.5%       16   18    9              0.89%
              4      2  most confident     1.1%    1.0%    1.0%       19   16   19                   
              8      1     all at once     0.0%    0.0%    0.0%        0    0    0              0.00%
   1, 1, then 6      3   left to right    98.8%   98.0%   98.1%       90   90   90
   1, 1, then 6      3          random    55.1%   56.5%   54.8%       90   90   90
digits 1 apart: 9 of the 9 possible differences fix the step
digits 2 apart: 1 of the 5 possible differences fix the step
digits 3 apart: 9 of the 9 possible differences fix the step
digits 4 apart: 1 of the 5 possible differences fix the step
digits 5 apart: 0 of the 2 possible differences fix the step
digits 6 apart: 1 of the 5 possible differences fix the step
digits 7 apart: 9 of the 9 possible differences fix the step
```

## The arithmetic: FLOPs and latency

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/diffusion-for-images-and-text/#the-arithmetic-flops-and-latency)

In [ ]:
# Generating L = 512 tokens with an 8B-parameter model on one H100 (989
# TFLOPS dense BF16, 3.35 TB/s), batch 1, ignoring attention's own cost.
# A forward pass costs about 2N FLOPs per position it processes, and reads
# all 2N bytes of BF16 weights once whatever the number of positions.
#   Autoregressive: L passes of 1 position each (the KV cache keeps earlier
#   positions). Each pass is memory-bound.
#   Masked diffusion: S passes over all L positions, because every position
#   can change at every step and a bidirectional model can't cache them.
N, L = 8e9, 512
flops_rate, bw = 989e12, 3.35e12
def step_time(positions):          # the slower of reading the weights and doing the arithmetic
    return max(2 * N / bw, 2 * N * positions / flops_rate)
ar_flops, ar_time = 2 * N * L, L * step_time(1)
print(f"autoregressive: {L} steps, {ar_flops / 1e12:6.1f} TFLOP, at least {ar_time:5.2f} s")
print(f"{'diffusion steps S':>18s} {'tokens per step':>16s} {'TFLOP':>8s} {'x AR FLOPs':>11s} {'time at least':>14s}")
for S in (512, 256, 128, 64, 32, 16):
    f = 2 * N * L * S
    print(f"{S:18d} {L / S:16.0f} {f / 1e12:8.0f} {f / ar_flops:11.0f} {S * step_time(L):12.2f} s")

The post printed:

```text
autoregressive: 512 steps,    8.2 TFLOP, at least  2.45 s
 diffusion steps S  tokens per step    TFLOP  x AR FLOPs  time at least
               512                1     4194         512         4.24 s
               256                2     2097         256         2.12 s
               128                4     1049         128         1.06 s
                64                8      524          64         0.53 s
                32               16      262          32         0.27 s
                16               32      131          16         0.13 s
```